### Data Ingestion

In [2]:
### Document data ingestion

from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content = "This is the main text content I am using to create RAG",
    metadata = {
        "source":"employee.txt",
        "page":1,
        "author":"Krish Naik",
        "date_created" : "2021-01-01"
    }
)
doc

Document(metadata={'source': 'employee.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2021-01-01'}, page_content='This is the main text content I am using to create RAG')

In [4]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [ ]:
### TextLoader
# from langchain.document_loaders import TextLoader ## This is the old import statement, now we need to use the new one as below

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [8]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [9]:
dir_loader

In [10]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20260312164027', 'source': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'file_path': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260312164027', 'page': 0}, page_content="Introduction to Coffee Culture\n1. Origins of Coffee\nCoffee was discovered in the Ethiopian highlands. Legend has it that a goat herder named Kaldi noticed his\ngoats became energetic after eating berries from a certain tree. This led to the cultivation of coffee beans,\nwhich eventually spread to the Arabian Peninsula and then worldwide.\n2. Regional Varieties\nEthiopia: Known for Yirgacheffe beans, which offer floral and citrus notes.\nBrazil: The world's largest coffee producer, known for nutty and chocolatey flavor profiles.\nVietnam: Fa

In [11]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Embedding and vector  store DB

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer #This is a library for generating sentence embeddings, which are dense vector representations of sentences. It is built on top of the Hugging Face Transformers library and provides an easy-to-use interface for generating embeddings for various natural language processing tasks.
import chromadb
from chromadb.config import Settings
import uuid #This is a library for generating universally unique ids
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\Dell\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initializes the embedding manager
        
        Args:
            model_name: Hugging Face model name for generating sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # protectded method to load the model
    
    def _load_model(self):
        """Loads the SentenceTransformer model"""
        try:
            print(f"⏳ Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ Loaded embedding model: {self.model_name} and embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise
    
    def generate_embeddings(self,documents:List[str]) -> np.ndarray:
        """Generates embeddings for a list of documents
        
        Args:
            texts: List of document texts to generate embeddings for.
        Returns:
            Numpy array of shape (num_documents, embedding_dimension) containing the generated embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        try:
            print(f"⏳ Generating embeddings for {len(documents)} documents...")
            embeddings = self.model.encode(documents, show_progress_bar=True)
            print(f"✅ Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"❌ Error generating embeddings: {e}")
            raise
        
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

⏳ Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2037.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded embedding model: all-MiniLM-L6-v2 and embedding dimension: 384


### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a chromadb vector store"""
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the chromadb collection to store embeddings.
            persist_directory: Directory where the chromadb database will be persisted in the hard disk.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._initialize_vector_store() # protected method to initialize the vector store
        
    def _initialize_vector_store(self):
        """Initializes the chromadb vector store and collection"""
        try:
            # create chromadb client with persistence settings
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) # create a persistent client that will save the vector store to disk
            
            # Get or create a collection for storing document embeddings
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Pdf document embeddings for RAG system"}
            )
            print(f"✅ Initialized vector store with collection: {self.collection_name} and persistence directory: {self.persist_directory}")
            print(f"Current number of embeddings in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")
            raise